# Geocode the Oct-2026 ECDA centre list by postal code (superseded)

**What this was for.** The first 2026 centre layer. The 2021 ChildCareServices GeoJSON was found to
be stale (292 active centres had no 2021 point; 541 of the 2021 points no longer appeared in the
2026 ECDA list), so the Oct-2026 ECDA listing was geocoded by postal code with the OneMap search
API, reusing 2021 coordinates where the postal code already had a point. Only child care centres
and kindergartens (`service_model` CC and KN, 1,651 centres) were kept. Output:
`data/Centres_2026.geojson`.

**Why it was superseded** (decision 2026-11-01, `docs/decisions.md`). Jessie's layer
([`../1_Data_Preparation/1_05_build_childcare_centres.ipynb`](../1_Data_Preparation/1_05_build_childcare_centres.ipynb))
keeps all service models: the 145 DS / EYC / EYC-DS centres this notebook dropped all offer full-day
care with a listed Nursery fee, and leaving them out understated supply in Fernvale, Yishun, Punggol
and Tampines East (children with no centre within 10 min: 612 vs 334). It also joins fees and keeps
a cache so reruns need no network. `data/Centres_2026.geojson` is deprecated and must not be used
for new work.

**Not re-executed.** It makes live OneMap calls (about one every 2.2 s without a token, roughly
10 minutes) and would overwrite the deprecated file. Original script:
`scripts/geocode_centres_2026.py` (Minh, 2026-10-28).

**Inputs**
- `data/CentreAttributes_2026.csv`, `data/ChildCareServices_2021.geojson`.

**Outputs**
- `data/Centres_2026.geojson` (EPSG:4326; deprecated) and, if any centre failed,
  `output/centres_2026_ungeocoded.csv`.

In [ ]:
import sys
from pathlib import Path

import os
import time

import geopandas as gpd
import pandas as pd
import requests

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

# changed 2026-11-10: paths through bt4015.paths.
SRC = DATA / "CentreAttributes_2026.csv"
OLD_POINTS = DATA / "ChildCareServices_2021.geojson"  # reused for shared postal codes
OUT_GEOJSON = DATA / "Centres_2026.geojson"
KEEP_MODELS = {"CC", "KN"}       # the restriction later reversed (DS / EYC centres also count as supply)
URL = "https://www.onemap.gov.sg/api/common/elastic/search"
HEADERS = {"User-Agent": "curl/8.4.0"}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)
if os.environ.get("ONEMAP_TOKEN"):
    SESSION.headers["Authorization"] = os.environ["ONEMAP_TOKEN"]

## 1. OneMap lookup with retries

A postal-code search only accepts a result in that postal code. Unauthenticated OneMap returns 429
after about one call every two seconds, so a 429 waits and retries.

In [ ]:
def geocode(postal, retries=6):
    """(lat, lon, OneMap address, status) for a six-digit postal code."""
    for attempt in range(retries):
        try:
            r = SESSION.get(URL, params={"searchVal": postal, "returnGeom": "Y",
                                         "getAddrDetails": "Y", "pageNum": 1}, timeout=(5, 10))
            if r.status_code == 200:
                res = [x for x in r.json().get("results", []) if x.get("POSTAL") == postal]
                if res:
                    x = res[0]
                    return float(x["LATITUDE"]), float(x["LONGITUDE"]), x["ADDRESS"], "ok"
                return None, None, None, "no_result"
            if r.status_code == 429:  # unauthenticated OneMap allows ~1 call / 2 s
                time.sleep(4 + 3 * attempt); continue
            return None, None, None, f"http_{r.status_code}"
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    return None, None, None, "error"

## 2. Active CC / KN centres and their postal codes

In [ ]:
# 1. One row per centre_code, CC and KN only (the later canonical layer keeps every model).
c = pd.read_csv(SRC)
c = c[c["service_model"].isin(KEEP_MODELS)].drop_duplicates("centre_code").copy()
c["postal_code"] = c["postal_code"].astype(str).str.zfill(6)
cols = ["centre_code", "centre_name", "organisation_description", "service_model",
        "scheme_type", "centre_address", "postal_code", "weekday_full_day",
        "government_subsidy", "spark_certified", "last_updated"]
c = c[cols].rename(columns={"organisation_description": "operator"})
uniq = c["postal_code"].unique()

# 2. Reuse 2021 coordinates for postal codes already seen there.
old = gpd.read_file(OLD_POINTS).to_crs(4326)
old["pc"] = old["ADDRESSPOSTALCODE"].astype(str).str.zfill(6)
old = old.drop_duplicates("pc").set_index("pc")
cache = {p: (old.at[p, "geometry"].y, old.at[p, "geometry"].x, "reused 2021 point", "ok_2021")
         for p in uniq if p in old.index}
todo = [p for p in uniq if p not in cache]
print(f"{len(c)} active CC/KN centres, {len(uniq)} unique postal codes: "
      f"{len(cache)} reused from 2021 points, {len(todo)} to geocode via OneMap", flush=True)

## 3. OneMap for the rest (live calls, throttled)

In [ ]:
t0 = time.time()
for i, p in enumerate(todo, 1):
    cache[p] = geocode(p)
    time.sleep(2.2)
    if i % 25 == 0:
        ok = sum(v[3].startswith("ok") for v in cache.values())
        print(f"  {i}/{len(todo)} done, {ok} ok, {time.time() - t0:.0f}s", flush=True)
c["lat"] = c["postal_code"].map(lambda p: cache[p][0])
c["lon"] = c["postal_code"].map(lambda p: cache[p][1])
c["onemap_address"] = c["postal_code"].map(lambda p: cache[p][2])
c["geocode_status"] = c["postal_code"].map(lambda p: cache[p][3])
print(c["geocode_status"].value_counts().to_string())

## 4. Write the (now deprecated) GeoJSON

In [ ]:
g = gpd.GeoDataFrame(c, geometry=gpd.points_from_xy(c["lon"], c["lat"]), crs="EPSG:4326")
g.to_file(OUT_GEOJSON, driver="GeoJSON")
print(f"Wrote {OUT_GEOJSON.relative_to(ROOT)}: {len(g)} centres, {int(g.geocode_status.str.startswith('ok').sum())} with coordinates")
bad = g[~g["geocode_status"].str.startswith("ok")]
if len(bad):
    bad.drop(columns="geometry").to_csv(OUT / "centres_2026_ungeocoded.csv", index=False)
    print(f"{len(bad)} centres without coordinates listed in output/centres_2026_ungeocoded.csv")